In [2]:
import re
import torch
from difflib import SequenceMatcher
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import GRPOConfig, GRPOTrainer

In [3]:
#Raw Data

In [4]:
data = [
    {
        "prompt": "Explain reinforcement learning in simple terms.",
        "reference_answer": "Reinforcement learning is a type of machine learning where an agent learns by taking actions, receiving rewards or penalties, and improving over time.",
    },
    {
        "prompt": "Write a polite email for delayed delivery.",
        "reference_answer": "We apologize for the delay in your delivery. Your order is on the way, and we appreciate your patience and understanding.",
    },
    {
        "prompt": "What is a neural network?",
        "reference_answer": "A neural network is a machine learning model inspired by the human brain. It learns patterns from data using layers of connected nodes.",
    },
    {
        "prompt": "Explain overfitting in ML.",
        "reference_answer": "Overfitting happens when a model learns the training data too closely, including noise, and performs poorly on new unseen data.",
    },
    {
        "prompt": "Explain supervised learning in simple terms.",
        "reference_answer": "Supervised learning is a machine learning method where a model learns from labeled examples and then predicts outputs for new data.",
    },
    {
        "prompt": "What is gradient descent?",
        "reference_answer": "Gradient descent is an optimization method that helps a model reduce its error by slowly adjusting its parameters in the right direction.",
    },
    {
        "prompt": "Explain classification in machine learning.",
        "reference_answer": "Classification is a machine learning task where the model assigns input data to predefined categories or classes.",
    },
    {
        "prompt": "Write a simple apology message.",
        "reference_answer": "I am sorry for the inconvenience. Thank you for your patience and understanding.",
    },
]

In [5]:
def format_prompt(example):
    return {
        "prompt": f"Question: {example['prompt']}\nAnswer in simple English:"
    }

In [6]:
print(format_prompt(data[0]))

{'prompt': 'Question: Explain reinforcement learning in simple terms.\nAnswer in simple English:'}


In [7]:
dataset = Dataset.from_list(data)

In [8]:
dataset = dataset.map(format_prompt) #function calling

Map: 100%|██████████| 8/8 [00:00<00:00, 3007.48 examples/s]


In [9]:
dataset

Dataset({
    features: ['prompt', 'reference_answer'],
    num_rows: 8
})

In [10]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1724.07it/s]


In [13]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

In [14]:
#Text convert to lowercase and remove extra spaces
def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [15]:
print(normalize_text("  Hello   World!  "))

hello world!


In [16]:
#This function safely extracts the answer text from either a normal string or chat-message format, and converts unknown formats into a string.
def get_completion_text(completion):
    """
    Works for normal string completions.
    Also safe if completion comes in chat/message format.
    """
    if isinstance(completion, str):
        return completion

    if isinstance(completion, list):
        try:
            return completion[-1]["content"]
        except Exception:
            return str(completion)

    return str(completion)

In [17]:
#sequence matcher is used to calculate the similarity score 
def similarity_score(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

In [18]:
# Example:

a = "Machine learning uses data"
b = "Machine learning learns from data"

# Common words: machine, learning, data

# So the score is:

# 3 common words ÷ 5 reference words = 0.6

def keyword_overlap_score(a: str, b: str) -> float:
    a_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(a)))
    b_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(b)))

    if not b_words:
        return 0.0

    return len(a_words & b_words) / len(b_words)

In [19]:
keyword_overlap_score(a, b)

0.6

In [20]:
# It acts like a teacher that compares the model’s answer with the answer key and gives marks out of 5.
def correctness_reward(prompts, completions, reference_answer=None, **kwargs):
    """
    Rewards answer similarity with reference answer.
    This is okay for demo, but not ideal for real production RL.
    """
    rewards = []

    if reference_answer is None:
        return [0.0 for _ in completions]

    for completion, ref in zip(completions, reference_answer):
        text = get_completion_text(completion)

        sim = similarity_score(text, ref)
        overlap = keyword_overlap_score(text, ref)

        # Combined correctness reward: 0 to 5
        score = (2.5 * sim) + (2.5 * overlap)
        rewards.append(float(score))

    return rewards

In [21]:
correctness_reward(
    prompts=[data[0]["prompt"]],
    completions=["Reinforcement learning is a type of machine learning where an agent learns by taking actions, receiving rewards or penalties, and improving over time."],
    reference_answer=[data[0]["reference_answer"]],
)

[5.0]

In [22]:
# It acts like a teacher who rewards complete and properly written answers and penalizes short, vague, or unhelpful answers.
def helpfulness_reward(prompts, completions, **kwargs):
    """
    Rewards answers that look useful and complete.
    Penalizes vague/unhelpful responses.
    """
    rewards = []

    bad_phrases = [
        "i don't know",
        "no idea",
        "maybe",
        "not sure",
        "random magic",
    ]

    for completion in completions:
        text = get_completion_text(completion)
        text_norm = normalize_text(text)

        score = 0.0

        # Good answer length
        if 40 <= len(text) <= 260:
            score += 1.0
        elif len(text) < 20:
            score -= 1.0
        elif len(text) > 350:
            score -= 0.5

        # Sentence-like response
        if "." in text or "," in text:
            score += 0.5

        # Penalize vague responses
        if any(bp in text_norm for bp in bad_phrases):
            score -= 1.5

        rewards.append(float(score))

    return rewards

In [23]:
helpfulness_reward(
    prompts=[data[0]["prompt"]],
    completions=["Reinforcement learning is a type of machine learning where an agent learns by taking actions, receiving rewards or penalties, and improving over time."],
)

[1.5]

In [24]:
# It acts like a teacher who rewards readable, properly sized answers and penalizes unclear or repetitive output.
def clarity_reward(prompts, completions, **kwargs):
    """
    Rewards readable English-style answers.
    """
    rewards = []

    for completion in completions:
        text = get_completion_text(completion)
        text_norm = normalize_text(text)

        score = 0.0

        # Has alphabetic content
        if re.search(r"[A-Za-z]", text):
            score += 0.5

        # Not too short / not too long
        if 30 <= len(text) <= 280:
            score += 0.5
        else:
            score -= 0.5

        # Penalize repeated junk
        if re.search(r"(.)\1{8,}", text_norm):
            score -= 1.0

        rewards.append(float(score))

    return rewards

In [25]:
clarity_reward(
    prompts=[data[0]["prompt"]],
    completions=["Reinforcement learning is a type of machine learning where an agent learns by taking actions, receiving rewards or penalties, and improving over time."],
)

[1.0]

In GRPO, the model does not directly learn from a fixed label like SFT. It generates multiple answers, and then reward functions score those answers. This code is only creating those reward functions: correctness checks similarity with the reference answer, helpfulness checks whether the answer is useful, and clarity checks whether the answer is readable. GRPOTrainer uses these scores to improve the model.

In GRPO, reward function is the teacher signal. For a demo, we can manually write simple rewards like correctness, helpfulness, and clarity. But in real systems, we do not keep writing unlimited reward functions manually. We usually use verifiable rewards, test cases, exact answer matching, LLM-as-judge, or a trained reward model. GRPO only needs a reward score; how we produce that reward depends on the task

In [26]:
#Init with Training
training_args = GRPOConfig(
    output_dir="grpo_output",

    learning_rate=1e-5,

    # Important:
    # In TRL GRPO, per_device_train_batch_size should be divisible by num_generations.
    # So we keep both as 4 for simple single-GPU Colab demo.
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_generations=4,

    max_completion_length=80,

    temperature=0.9,
    top_p=0.95,

    # KL penalty. 0.0 is default in recent TRL GRPO, but 0.01 is okay for teaching demo.
    beta=0.01,

    max_steps=10,

    logging_steps=1,
    save_steps=10,

    remove_unused_columns=False,
    report_to="none",

    gradient_checkpointing=False,

    fp16=torch.cuda.is_available(),
)

In [27]:
print(torch.cuda.is_available())

True


In [29]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward,
        helpfulness_reward,
        clarity_reward,
    ],
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
)

In [30]:
import trl
import inspect
from trl import GRPOConfig

print("TRL version:", trl.__version__)
print(inspect.signature(GRPOConfig.__init__))

TRL version: 1.14.2
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 1e-06, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: boo

In [31]:
trainer.train()

Step,Training Loss
1,-0.000000
2,-0.000001
3,-0.000000
4,0.023565
5,0.140240
6,0.000000
7,0.000000
8,0.000001
9,-0.000000
10,0.000000


TrainOutput(global_step=10, training_loss=0.01638056106865917, metrics={'train_runtime': 50.8502, 'train_samples_per_second': 0.787, 'train_steps_per_second': 0.197, 'total_flos': 0.0, 'train_loss': 0.01638056106865917, 'epoch': 1.25})

In [35]:
import os
os.getcwd()

'c:\\Users\\SiddheshLihe\\Transformer-LLM-Training-Tuning-Reasoning\\Transformer_notebook'

In [34]:
trainer.save_model("./grpo_output/grpo_output_final")
tokenizer.save_pretrained("./grpo_output/grpo_output_final")

('./grpo_output/grpo_output_final\\tokenizer_config.json',
 './grpo_output/grpo_output_final\\chat_template.jinja',
 './grpo_output/grpo_output_final\\tokenizer.json')

In [36]:
base_model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
adapter_path = "./grpo_output/grpo_output_final"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1063.39it/s]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 576, padding_idx=2)
        (layers): ModuleList(
          (0-29): 30 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=576, out_features=576, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=576, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=576, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [37]:
def generate_answer(question, max_new_tokens=100):
    prompt = f"Question: {question}\nAnswer in simple English:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [38]:
print(generate_answer("Explain AI in simple terms."))

Question: Explain AI in simple terms.
Answer in simple English:

**What is Artificial Intelligence (AI)?**

Artificial Intelligence, or AI, is a type of technology that enables computers to perform tasks that typically require human intelligence, such as:

* Learning from data
* Solving problems
* Making decisions
* Understanding natural language
* Recognizing images and speech

**How does AI work?**

AI systems use algorithms and data to analyze and interpret information. Here's a step-by-step explanation:

1. **
